# Sprint 3 - Tools and MCP for HelioDesk

This Colab is the shared Sprint 3 notebook for the HelioDesk support assistant. It turns the sprint concepts into one inspectable application path: a support specialist needs to decide whether a customer export can be shared with an external auditor.

You will work through three connected tasks:

1. inspect and run a direct authorization tool;
2. classify tool failures and choose safe recovery patterns; and
3. compare a multi-step tool loop with an MCP policy-search boundary.

Most implementation details are hidden behind helper functions so the notebook can stay focused on evidence, decisions, and checks. You can expand hidden cells if you are curious, but the student work happens in the visible cells.

Choose **File > Save a copy in Drive** before editing in Colab.

**Resource type:** Campus educational notebook: sprint-specific business case from the Campus lessons.


## 1. Install the helper core from GitHub

Run this first in Colab. It installs the shared helper core directly from the GitHub `main` branch so the notebook uses the same code as the course scaffold.


In [ ]:
#@title Install helper core from GitHub { display-mode: "form" }
%pip install -q --force-reinstall --no-cache-dir "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"

## 2. Set up the notebook

Store `OPENROUTER_API_KEY` in Colab Secrets, or use the hidden prompt. The tool loop requires a real model connection and spends API credits. Authorization records are the supplied course dataset, not a connection to a customer's live workspace.


In [ ]:
#@title Setup: imports and high-level notebook helpers { display-mode: "form" }
from sprint3_tools_mcp import (
    build_checkpoint_evidence,
    build_heliodesk_tool_registry,
    connect_to_heliodesk_policy_mcp,
    display_direct_authorization_check,
    display_failure_evidence,
    display_mcp_validation,
    display_recovery_review,
    display_registered_tools,
    display_tool_loop,
    inspect_and_display_contracts,
    pretty,
    review_decision_note,
    review_recovery_plan,
    run_direct_authorization_check,
    run_failure_scenarios,
    run_live_tool_loop,
    summarize_tool_trace,
    show_case_brief,
    validate_checkpoint_evidence,
    validate_mcp_policy_response,
    assert_direct_authorization_check,
    assert_failure_scenarios,
)

In [ ]:
from notebook_setup import require_openrouter_key

_ = require_openrouter_key(prompt=True)

## 3. Read the case brief

The same business question appears throughout the notebook. Keep the two evidence sources separate: current authorization state and handbook policy evidence.


In [ ]:
case_brief = show_case_brief()
pretty(case_brief)

## 4. Inspect the schemas sent to the model

A tool is application code plus a model-facing contract. The model may propose a call, but the application validates the arguments, executes the handler, and decides whether the result is safe to use.

Run the next two cells, then answer Task A from the outputs.


In [ ]:
registry = build_heliodesk_tool_registry()
display_registered_tools(registry)

In [ ]:
openrouter_tool_schemas, contract_report = inspect_and_display_contracts(registry)

## 5. Execute a valid direct tool call

This direct call checks current authorization for the HelioDesk case. Notice what the result proves, and what it does not prove.


In [ ]:
authorization_result, authorization_payload = run_direct_authorization_check(registry)
display_direct_authorization_check(authorization_result, authorization_payload)
assert_direct_authorization_check(authorization_result, authorization_payload)

## 6. Handle malformed and failed tool calls

Run invalid JSON, unsupported arguments, and an absent-record lookup through the actual registry. Inspect the errors it returns before choosing a recovery action. Timeouts and permission errors can occur with external services, but this local-data exercise does not manufacture them.


In [ ]:
failure_evidence, failure_raw = run_failure_scenarios(registry)
display_failure_evidence(failure_evidence, failure_raw, focus_case="empty_result")
assert_failure_scenarios(failure_evidence)

### Your turn: choose recovery patterns

Keep each recovery decision bounded. Do not treat missing, failed, or permission-blocked tool output as approval to continue.


In [ ]:
failure_decisions = {
    "invalid_input": "ask_for_missing_input",
    "malformed_response": "reject_and_log",
    "missing_data": "reject_and_log",
    "timeout": "retry_once_then_route",
    "empty_result": "honest_no_result",
    "permission_failure": "route_to_authorized_owner",
}

recovery_review = review_recovery_plan(failure_evidence, failure_decisions)
display_recovery_review(recovery_review)
assert all(row["passes"] for row in recovery_review)

## 7. Let the model run a multi-step tool loop

The LLM receives the user request and tool schemas. It chooses arguments, your Python registry executes the requested tools, and their results return to the model before it writes an answer. This cell makes real OpenRouter requests.

Inspect the returned trace: authorization must run before policy search. A tool schema describes an available action; it is not evidence that the action ran. The trace and returned data are your evidence.


In [ ]:
# The model returns tool-call requests; the registry executes real handlers.
model_run = run_live_tool_loop(registry)
tool_trace = summarize_tool_trace(model_run)
display_tool_loop(model_run, tool_trace)

## 8. Connect to the MCP server

Model Context Protocol, or MCP, is a standard connection boundary between an AI host and a server that exposes capabilities. This cell creates the helper-core MCP server object, then launches the installed helper server over `stdio` and calls its `keyword_search` tool with HelioDesk policy snippets.


In [ ]:
mcp_demo, mcp_summary = await connect_to_heliodesk_policy_mcp()
pretty(mcp_summary)

## 9. Validate the MCP response

An MCP connection proves only that the boundary exists. The host application still validates the advertised capability and returned content before using it downstream.


In [ ]:
validated_mcp_rows, mcp_downstream_note = validate_mcp_policy_response(mcp_demo)
display_mcp_validation(validated_mcp_rows, mcp_downstream_note)

assert validated_mcp_rows
assert any("authorization" in row["text"].lower() for row in validated_mcp_rows)

### Your turn: write the decision note

Before collecting final evidence, write your own short note answering Task C. Explain one contract decision, one failure-handling decision, and how the MCP result differs from the direct authorization lookup. Use evidence from the cells you ran; no sample answer is pre-filled.


In [ ]:
decision_note = ""  # Write at least 45 words from your own run.

decision_note_review = review_decision_note(decision_note)
pretty(decision_note_review)
if not decision_note_review["passes"]:
    print("Complete the decision note before validating the checkpoint.")

## 10. Sprint 3 checkpoint evidence

Record your Task A, B, and C explanations before comparing them with the answer signals. The final cell validates the runtime evidence immediately, but it keeps comparison guidance closed until all three written responses are substantive.


In [ ]:
TASK_RESPONSES = {
    "task_a_direct_tool_contract": "",
    "task_b_failure_handling": "",
    "task_c_multi_step_and_mcp": decision_note,
}
checkpoint_evidence = build_checkpoint_evidence(
    contract_report=contract_report,
    authorization_result=authorization_result,
    authorization_payload=authorization_payload,
    failure_evidence=failure_evidence,
    recovery_review=recovery_review,
    model_run=model_run,
    mcp_demo=mcp_demo,
    validated_mcp_rows=validated_mcp_rows,
    mcp_downstream_note=mcp_downstream_note,
    decision_note_review=decision_note_review,
)
pretty({"your_task_outputs": checkpoint_evidence})


def reveal_answer_signals():
    incomplete = [
        key for key, value in TASK_RESPONSES.items() if len(value.split()) < 30
    ]
    if incomplete or not decision_note_review["passes"]:
        print(
            "Complete all three evidence-based responses before comparison:",
            ", ".join(incomplete),
        )
        return
    from sprint3_tools_mcp import build_task_answer_signals

    pretty({"answer_signals": build_task_answer_signals()})


if decision_note_review["passes"]:
    validate_checkpoint_evidence(
        authorization_payload=authorization_payload,
        recovery_review=recovery_review,
        model_run=model_run,
        validated_mcp_rows=validated_mcp_rows,
        decision_note_review=decision_note_review,
    )
else:
    print(
        "Runtime evidence is present. Finish the written responses, then rerun this cell."
    )

## 11. FieldCare project checkpoint: direct tools and genuine MCP

Apply this sprint's boundary decision to FieldCare. Structured current-state operations stay as five direct `ToolRegistry` tools. Service-document search is exposed only by a separate MCP server. Run both boundaries, inspect the advertised MCP capability and actual result, then export the capability contract for Sprint 4.


In [ ]:
import json
from fieldcare import (
    build_fieldcare_direct_registry,
    download_fieldcare_assets,
    inspect_and_call_fieldcare_mcp,
    write_fieldcare_artifact,
)

FIELDCARE_ASSET_DIR = download_fieldcare_assets("fieldcare-assets")
fieldcare_direct_registry = build_fieldcare_direct_registry(FIELDCARE_ASSET_DIR)
fieldcare_direct_schemas = fieldcare_direct_registry.to_openrouter_tools()
fieldcare_equipment_check = fieldcare_direct_registry.execute_tool_call(
    {
        "id": "fieldcare-equipment-check",
        "function": {
            "name": "get_equipment_record",
            "arguments": json.dumps({"equipment_id": "EQ-FC-1001"}),
        },
    }
)
print([item["function"]["name"] for item in fieldcare_direct_schemas])
print(json.loads(fieldcare_equipment_check.content)["status"])

In [ ]:
fieldcare_mcp_demo = await inspect_and_call_fieldcare_mcp(
    query="HX overheating after filter replacement airflow",
    top_k=3,
    asset_dir=FIELDCARE_ASSET_DIR,
)
fieldcare_mcp_tool_names = [tool.name for tool in fieldcare_mcp_demo.tools]
fieldcare_mcp_rows = []
for text in fieldcare_mcp_demo.call.content_texts:
    parsed = json.loads(text)
    fieldcare_mcp_rows.extend(parsed if isinstance(parsed, list) else [parsed])
pretty({"advertised_mcp_tools": fieldcare_mcp_tool_names, "rows": fieldcare_mcp_rows})
assert fieldcare_mcp_tool_names == ["search_service_docs"]
assert fieldcare_mcp_rows

In [ ]:
direct_names = [item["function"]["name"] for item in fieldcare_direct_schemas]
FIELDCARE_CAPABILITY_CONTRACTS = {
    "schema_version": "1.0",
    "project_id": "fieldcare",
    "direct_tools": [
        {
            "name": item["function"]["name"],
            "description": item["function"]["description"],
        }
        for item in fieldcare_direct_schemas
    ],
    "mcp_server": {
        "name": fieldcare_mcp_demo.server_name,
        "tools": [{"name": name} for name in fieldcare_mcp_tool_names],
    },
    "orchestration_rules": [
        {
            "request_type": "troubleshooting_plus_warranty",
            "required_inputs": ["request_text", "equipment_id", "ticket_id"],
            "available_inputs_before_call": [
                "request_text",
                "equipment_id",
                "ticket_id",
            ],
            "call_order": [
                "search_service_docs",
                "get_equipment_record",
                "get_maintenance_history",
                "get_warranty_status",
                "get_ticket_status",
                "recommend_escalation_path",
            ],
        },
        {
            "request_type": "safety_escalation",
            "required_inputs": ["request_text", "equipment_id", "ticket_id"],
            "available_inputs_before_call": [
                "request_text",
                "equipment_id",
                "ticket_id",
            ],
            "call_order": [
                "search_service_docs",
                "get_equipment_record",
                "get_ticket_status",
                "recommend_escalation_path",
            ],
        },
        {
            "request_type": "ticket_status_only",
            "required_inputs": ["request_text", "equipment_id", "ticket_id"],
            "available_inputs_before_call": [
                "request_text",
                "equipment_id",
                "ticket_id",
            ],
            "call_order": ["get_ticket_status"],
        },
    ],
    "failure_policy": {
        "missing_input": "ask",
        "timeout": "retry_once_then_route",
        "unavailable": "abstain_and_route",
    },
    "trace_evidence": [
        {
            "boundary": "direct",
            "tool": "get_equipment_record",
            "status": json.loads(fieldcare_equipment_check.content)["status"],
        },
        {
            "boundary": "mcp",
            "tool": "search_service_docs",
            "doc_ids": [row["doc_id"] for row in fieldcare_mcp_rows],
        },
    ],
}
CAPABILITY_PATH = write_fieldcare_artifact(
    FIELDCARE_CAPABILITY_CONTRACTS,
    "capability_contracts",
    "fieldcare_capability_contracts.json",
)
print(f"Validated capability contracts: {CAPABILITY_PATH}")

In [ ]:
#@title Download the Sprint 3 project artifact { display-mode: "form" }
try:
    from google.colab import files

    files.download(str(CAPABILITY_PATH))
except ImportError:
    print(CAPABILITY_PATH)

## Optional local inspector

The notebook already connects to the helper MCP server over `stdio`. If you want to inspect the same server locally with the MCP Inspector, run this from a local clone:

```bash
git clone https://github.com/richhiey/ai-app-dev_Mod-A.git
cd ai-app-dev_Mod-A
python -m pip install -e ".[dev]"
mcp dev src/mcp_server.py
```

In the local inspector, call `keyword_search` with the HelioDesk policy snippets from this notebook. Treat the inspector output as UI evidence only after you have captured it from the actual tool.
